# Customer Churn Prediction: Model Training & Evaluation

This notebook trains and compares classification models for customer churn prediction. It documents the path from feature-engineered data to the saved production model in `models/churn_model.joblib`.

**Business goal:** identify customers likely to churn so the company can prioritize retention actions before revenue is lost.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")

## 2. Load and Prepare Data

The same cleaning logic from the EDA and feature engineering notebooks is repeated here so the model training workflow is reproducible end to end.

In [ ]:
DATA_PATH = Path("../data/raw/telco_customer_churn.csv")
MODEL_PATH = Path("../models/churn_model.joblib")

df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})

df.shape

In [ ]:
df[["TotalCharges", "ChurnFlag"]].isnull().sum()

## 3. Define Features and Target

`customerID` is dropped because it is an identifier. `Churn` and `ChurnFlag` are not used as input features because they represent the outcome.

In [ ]:
X = df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df["ChurnFlag"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features, categorical_features

In [ ]:
y.value_counts(normalize=True).mul(100).round(2)

### Class Imbalance Reminder

The churn class is the minority class. Because of this, accuracy is not enough. The model must be evaluated on its ability to correctly identify churners using recall, precision, F1 score, ROC-AUC, and the confusion matrix.

## 4. Train/Test Split

The split is stratified to preserve the churn/non-churn ratio in both training and testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

## 5. Preprocessing Pipeline

The preprocessing pipeline scales numeric features and one-hot encodes categorical features. It is placed inside each model pipeline so transformations are fit only on training data.

In [ ]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

## 6. Models to Compare

Three models are compared:

- Logistic Regression: interpretable baseline
- Decision Tree: simple non-linear model
- Random Forest: stronger ensemble model that can capture interactions

The objective is not to use the most complex model. The objective is to compare reasonable options and justify the final selection.

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Decision Tree": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=8,
                    min_samples_leaf=8,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

models

## 7. Train and Evaluate Models

In [ ]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }

In [ ]:
results = {}
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    results[model_name] = evaluate_model(model, X_test, y_test)
    trained_models[model_name] = model

results_df = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)
results_df.round(4)

### Model Comparison Interpretation

The best model should balance business usefulness and predictive performance. In churn prediction, recall is especially important because missing a likely churner means the business may lose the chance to intervene.

ROC-AUC is used as the primary selection metric because it measures how well the model separates churners from non-churners across probability thresholds. Recall and F1 are also reviewed because this is an intervention problem.

## 8. Select Best Model

In [ ]:
best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

best_model_name, results_df.loc[best_model_name].round(4).to_dict()

### Why This Model Wins

The selected model provides the strongest ROC-AUC while maintaining high recall. That makes it suitable for ranking customers by churn risk and helping retention teams decide who should be contacted first.

This model should not be treated as an automatic decision system. It should be used as a prioritization tool alongside customer lifetime value, contact policies, and retention budget.

## 9. Confusion Matrix

In [ ]:
best_predictions = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_predictions)

ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.show()

### Confusion Matrix Interpretation

The confusion matrix shows where the model is useful and where it makes mistakes:

- True positives: churners correctly flagged for retention action
- False negatives: churners missed by the model
- False positives: retained customers flagged as risky
- True negatives: retained customers correctly identified

For churn prevention, false negatives are especially costly because the business loses an opportunity to intervene.

## 10. ROC Curve

In [ ]:
best_probabilities = best_model.predict_proba(X_test)[:, 1]

RocCurveDisplay.from_predictions(y_test, best_probabilities)
plt.title(f"ROC Curve: {best_model_name}")
plt.show()

### ROC-AUC Interpretation

ROC-AUC evaluates how well the model ranks churners above non-churners. A stronger ROC-AUC means the model is better at separating high-risk customers from lower-risk customers before a final cutoff is chosen.

This is valuable because the business may choose different thresholds depending on retention budget and campaign capacity.

## 11. Feature Importance

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

if hasattr(classifier, "feature_importances_"):
    importance_values = classifier.feature_importances_
elif hasattr(classifier, "coef_"):
    importance_values = classifier.coef_[0]
else:
    importance_values = None

feature_importance = pd.DataFrame(
    {"feature": feature_names, "importance": importance_values}
)
feature_importance["abs_importance"] = feature_importance["importance"].abs()
feature_importance = feature_importance.sort_values("abs_importance", ascending=False)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).copy()
top_features["feature"] = (
    top_features["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

plt.figure(figsize=(8, 6))
sns.barplot(data=top_features, y="feature", x="abs_importance", color="#059669")
plt.title(f"Top Predictive Features: {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("")
plt.show()

### Feature Importance Interpretation

Feature importance helps compare the model's learned patterns against the earlier feature hypotheses. If contract type, tenure, monthly charges, payment method, and service/support fields rank highly, the model is consistent with the business EDA.

This step is important because a model should not only perform well; it should also produce patterns that can be explained to business stakeholders.

## 12. Save Final Model

The saved object includes the full preprocessing and model pipeline. This is what allows the Streamlit app to accept raw customer inputs and generate churn probabilities consistently.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": best_model,
        "model_name": best_model_name,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "metrics": results_df.loc[best_model_name].round(4).to_dict(),
    },
    MODEL_PATH,
)

MODEL_PATH

## 13. Business Summary

The final model is useful because it converts customer data into churn probabilities. These probabilities can support retention workflows such as:

- prioritizing high-risk customers for outreach
- targeting month-to-month customers with upgrade incentives
- monitoring new customers during the first 12 months
- designing payment migration campaigns for electronic check customers
- bundling support services for customers with service-related churn risk

The next notebook should translate model scores into retention strategy recommendations and business actions.